# Multi-Modal Tri-Stream AVNet & AdapterNet: Cloud GPU Training Pipeline

This notebook trains the **Tri-Stream AVNet (AVNet-Mag)** and **AdapterNet-9Axis** dead-reckoning models on Google Colab or Kaggle with GPU acceleration.

### Pipeline Overview:
1. **Environment Setup**: Detect GPU (Tesla T4 / V100 / A100) and install requirements.
2. **Dataset Setup**: Download / mount the 144 synchronized IO-VNBD dataset pairs (`S-*.csv` + `V-*.csv`).
3. **AVNet Training**: Train decoupled 3-stream CNNs + BiGRU + Attention Pooling with multi-task Huber & Geodesic loss.
4. **AdapterNet Training**: Optimize dynamic process/measurement noise covariances ($Q, N$).
5. **Trajectory Evaluation**: Run Lie-group InEKF dead reckoning and compute ATE, $E_{\text{trel}}$, $E_{\text{rrel}}$.
6. **Export Checkpoints**: Save model weights to `.pkl` and `.pth` files with 1-click download.

In [ ]:
# 1. GPU Check & Package Installation
!nvidia-smi
!pip install -q torch torchvision torchaudio numpy scipy pandas matplotlib tqdm

In [ ]:
# 2. Clone Repository or Set Workspace
import os, sys

# If running in Google Colab, clone the repo:
if not os.path.exists('avnet'):
    print("Cloning repository...")
    # Replace with your actual GitHub repository URL:
    # !git clone https://github.com/YOUR_USERNAME/avnet.git
    # os.chdir('avnet')
    pass

sys.path.append('.')
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using compute device: {device}")

In [ ]:
# 3. Download & Extract IO-VNBD Dataset
# The dataset can be downloaded directly, or mounted from Google Drive

import os
data_dir = 'data'

# Example for Google Colab Drive Mount:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/IO-VNBD data/

if not os.path.exists(data_dir):
    print(f"Creating {data_dir} directory. Please ensure dataset CSV files are placed here.")
    os.makedirs(data_dir, exist_ok=True)

print(f"Dataset root: {os.path.abspath(data_dir)}")

In [ ]:
# 4. Verify Paired Dataset Files (S-*.csv paired with V-*.csv)
from avnet.dataset import discover_paired_iovnbd_files

pairs = discover_paired_iovnbd_files(data_dir)
print(f"Total discovered pairs: {len(pairs)}")
if pairs:
    print(f"Sample S-file: {pairs[0][0]}")
    print(f"Sample V-file: {pairs[0][1]}")

In [ ]:
# 5. Create Train / Validation / Test DataLoaders
from avnet.dataset import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(
    root_dir=data_dir,
    window_size=20,     # 2.0 seconds at 10 Hz
    step=2,             # 90% overlap for dense training signals
    batch_size=64,      # Optimized for GPU
    train_ratio=0.8,
    val_ratio=0.1,
    limit_files=None    # Set to integer (e.g. 10) for faster test runs
)

batch = next(iter(train_loader))
print("Sample Batch Shapes:")
print("  Accelerometer Stream:", batch['acc'].shape)   # (B, 3, 20)
print("  Gyroscope Stream:    ", batch['gyro'].shape)  # (B, 3, 20)
print("  Magnetometer Stream: ", batch['mag'].shape)   # (B, 3, 20)
print("  Target Forward Speed:", batch['target_speed'].shape) # (B, 1)
print("  Target Delta Quat:   ", batch['target_delta_q'].shape) # (B, 3)

In [ ]:
# 6. Train Multi-Modal Tri-Stream AVNet
from avnet.models.avnet import TriStreamAVNet
from avnet.train import train_tristream_avnet

model = TriStreamAVNet(window_size=20, hidden_dim=64)

# Hyperparameters
EPOCHS = 15
LEARNING_RATE = 1e-3
LAMBDA_ATT = 5.0  # Higher weight on attitude to eliminate heading drift

trained_avnet, history = train_tristream_avnet(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    lambda_att=LAMBDA_ATT,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 7. Plot Training and Validation Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='#2563eb', lw=2)
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val Loss', color='#dc2626', lw=2)
axes[0].set_title('Multi-Task Loss (Huber + Geodesic)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speed RMSE
if history['speed_rmse']:
    axes[1].plot(history['speed_rmse'], label='Speed RMSE (m/s)', color='#16a34a', lw=2)
    axes[1].set_title('Validation Speed RMSE (m/s)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('m/s')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Attitude Error
if history['att_error_deg']:
    axes[2].plot(history['att_error_deg'], label='Attitude Error (deg)', color='#9333ea', lw=2)
    axes[2].set_title('Validation Attitude Geodesic Error (°)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Degrees')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=300)
plt.show()

In [ ]:
# 8. Train AdapterNet-9Axis (Offline Indirect Trajectory Optimization)
from avnet.models.avnet import AdapterNet9Axis
from avnet.train_adapter import train_adapter_offline
from avnet.dataset import load_iovnbd_csv

print("Loading sequences for AdapterNet offline optimization...")
sample_sequences = [load_iovnbd_csv(s, v) for s, v in pairs[:5] if v is not None]

adapter_model = AdapterNet9Axis(in_channels=9)
trained_adapter = train_adapter_offline(
    adapter_model=adapter_model,
    avnet_model=trained_avnet,
    data_sequences=sample_sequences,
    epochs=5,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 9. Closed-Loop InEKF Trajectory Benchmark
from main import run_evaluation

test_eval_data = load_iovnbd_csv(pairs[0][0], pairs[0][1])
results = run_evaluation(
    avnet_model=trained_avnet,
    adapter_model=trained_adapter,
    eval_data=test_eval_data,
    device=device,
    output_dir='results'
)

# Plot Estimated vs Ground Truth Path
pred_p = results['pred_positions']
gt_p = test_eval_data['gt_enu']

plt.figure(figsize=(10, 8))
plt.plot(gt_p[:, 0], gt_p[:, 1], 'r--', label='Ground Truth (Vehicle CAN/GPS)', lw=2.5)
plt.plot(pred_p[:, 0], pred_p[:, 1], 'b-', label='DMDVDR InEKF Dead Reckoning', lw=2.0)
plt.scatter(gt_p[0, 0], gt_p[0, 1], c='green', s=100, zorder=5, label='Start Point')
plt.title(f"Trajectory Dead Reckoning: Etrel={results['rel_metrics']['E_trel_percent']:.2f}% | ATE={results['ate']:.2f}m")
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.savefig('results/trajectory_benchmark.png', dpi=300)
plt.show()

In [ ]:
# 10. Package & Save Trained Model to .pkl and .pth Files
import pickle

# Package complete model checkpoint for inference
model_export_pkl = 'checkpoints/avnet_tristream_model.pkl'
export_package = {
    'model_name': 'TriStreamAVNet-Mag',
    'window_size': 20,
    'hidden_dim': 64,
    'state_dict': trained_avnet.state_dict(),
    'adapter_state_dict': trained_adapter.state_dict(),
    'final_metrics': {
        'val_loss': history['val_loss'][-1] if history['val_loss'] else None,
        'speed_rmse': history['speed_rmse'][-1] if history['speed_rmse'] else None,
        'att_deg': history['att_error_deg'][-1] if history['att_error_deg'] else None
    }
}

with open(model_export_pkl, 'wb') as f:
    pickle.dump(export_package, f)

print(f"Successfully exported model package to: {model_export_pkl}")
print(f"File size: {os.path.getsize(model_export_pkl) / 1024:.2f} KB")

In [ ]:
# 11. Download Model Artifact Directly to Local Machine (Colab Helper)
try:
    from google.colab import files
    print("Triggering browser download for trained model file...")
    files.download('checkpoints/avnet_tristream_model.pkl')
    files.download('checkpoints/best_avnet_tristream.pth')
    files.download('results/calibration_profile.json')
except ImportError:
    print("Not running in Google Colab. Model saved locally in checkpoints/avnet_tristream_model.pkl")